<a href="https://colab.research.google.com/github/aicuai/Book-StartGuideSDXL/blob/main/AICU_Civitai_LoRA_Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/aicuai/Book-StartGuideSDXL/blob/main/AICU_Civitai_LoRA_Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AICU Civitai LoRA Trainer `v.260611.2`

「画像・動画生成AI スタートガイド」第3-11節 連動コンテンツをベースに
新刊書籍「SG26」に登場するキャラクター 響姫メイ（HibikiMei） のオリジナル LoRA を Kohya `sd-scripts` で学習するノートブックです。

Civitai のモデルをベースモデルとして使用します。

- ベースモデル: Sierunami v1（Illustrious License）https://j.aicu.ai/sierunami
- Kohya sd-scripts: v0.10.5 固定
- PyTorch: 2.6.0 + CUDA 12.4


## 1. 環境とシークレットの設定

Colab の GPU ランタイムを使ってください。最初に依存関係を入れ替えたあと、`torch` がすでに読み込まれていた場合はランタイム再起動が必要になる場合があります。


> Colab 左側の 🔑 から `CIVITAI_KEY`（Civitai API Key）を設定してください。


In [2]:
#@title 1.1 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

for d in [DATASET_DIR, MODEL_DIR, OUTPUT_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories ready')


Mounted at /content/drive
Directories ready


In [3]:
#@title 1.1 Load CIVITAI_KEY
from google.colab import userdata

CIVITAI_KEY = userdata.get("CIVITAI_KEY")
if not CIVITAI_KEY:
    raise ValueError(
        "CIVITAI_KEY が Colab Secrets にありません\n"
        "https://civitai.com/user/account で取得してください"
    )

print("CIVITAI_KEY: loaded from Colab Secrets")

CIVITAI_KEY: loaded from Colab Secrets


In [ ]:
#@markdown ### 📂 1.1 データセットの準備
#@markdown あなたの Google Drive に接続し、サンプルデータセットをダウンロードして展開します。
#@markdown 自分のデータセットを使う場合は extract_to のフォルダに PNG を入れてください。
import os, zipfile, urllib.request

download_from = "https://huggingface.co/AICU/SDXL-LoRA/resolve/main/HibikiMei.zip"  #@param {type:"string"}
zip_path      = "/content/drive/MyDrive/Loras/HibikiMei.zip"                         #@param {type:"string"}
extract_to    = "/content/drive/MyDrive/Loras/HibikiMei/dataset"                     #@param {type:"string"}

os.makedirs(extract_to, exist_ok=True)

if not os.path.exists(zip_path):
    print(f"Downloading {download_from} ...")
    urllib.request.urlretrieve(download_from, zip_path)
    print("Download complete.")
else:
    print(f"Already downloaded: {zip_path}")

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_to)

png_count = len([f for f in os.listdir(extract_to) if f.endswith(".png")])
print(f"Dataset ready: {png_count} PNG files in {extract_to}")


In [1]:
#@title 2 Project settings
from pathlib import Path

# --- プロジェクト設定 ---
PROJECT_NAME  = "HibikiMei"  #@param {type:"string"}
TRIGGER_WORDS = "HibikiMei"  #@param {type:"string"}

DRIVE_PROJECT_DIR = Path(f"/content/drive/MyDrive/Loras/{PROJECT_NAME}")
DATASET_DIR = DRIVE_PROJECT_DIR / "dataset"
MODEL_DIR   = DRIVE_PROJECT_DIR / "models"
OUTPUT_DIR  = DRIVE_PROJECT_DIR / "output"
CONFIG_DIR  = DRIVE_PROJECT_DIR / "config"

# --- ベースモデル選択 ---
BASE_MODEL = "Sierunami v1"  #@param ["Sierunami v1", "Animagine XL 4.0", "Custom"]
CUSTOM_MODEL_URL = ""  #@param {type:"string"}

_MODEL_URLS = {
    "Sierunami v1":     "https://civitai.com/api/download/models/1176276",
    "Animagine XL 4.0": "https://huggingface.co/cagliostrolab/animagine-xl-4.0/resolve/main/animagine-xl-4.0.safetensors",
}
BASE_MODEL_NAME = BASE_MODEL.replace(" ", "_")
BASE_MODEL_URL  = CUSTOM_MODEL_URL if BASE_MODEL == "Custom" else _MODEL_URLS[BASE_MODEL]
BASE_MODEL_PATH = MODEL_DIR / f"{BASE_MODEL_NAME}.safetensors"

# --- Kohya LTS ---
KOHYA_REPO = "https://github.com/kohya-ss/sd-scripts.git"
KOHYA_REF  = "v0.10.5"  #@param {type:"string"}
KOHYA_DIR  = Path("/content/sd-scripts")

# --- PyTorch (CUDA 12.4 wheel) ---
PYTORCH_VERSION     = "2.6.0"
TORCHVISION_VERSION = "0.21.0"
CUDA_INDEX_URL      = "https://download.pytorch.org/whl/cu124"

# --- 学習パラメータ ---
RESOLUTION       = "1024,1024"  #@param {type:"string"}
NUM_REPEATS      = 10   #@param {type:"integer"}
TRAIN_BATCH_SIZE = 1    #@param {type:"integer"}
MAX_TRAIN_EPOCHS = 10   #@param {type:"integer"}
NETWORK_DIM      = 32   #@param {type:"integer"}
NETWORK_ALPHA    = 16   #@param {type:"integer"}
LEARNING_RATE    = "1e-4"  #@param {type:"string"}

print("Project:", PROJECT_NAME)
print("Base model:", BASE_MODEL, "→", BASE_MODEL_URL)
print("Dataset:", DATASET_DIR)
print("Kohya:", KOHYA_REF)


Project: HibikiMei
Base model: Sierunami v1 → https://civitai.com/api/download/models/1176276
Dataset: /content/drive/MyDrive/Loras/HibikiMei/dataset
Kohya: v0.10.5


## 3. Download Sierunami v1

Civitai から Sierunami v1 をダウンロードします。Drive に保存するのでセッションが切れても再ダウンロード不要です。

Colab 左側の 🔑 から `CIVITAI_KEY` を設定してください。

- Name: `CIVITAI_KEY`
- Value: https://civitai.com/user/account で発行した API Key


In [5]:
#@title 3 Download Sierunami v1
# MODEL_DIR が未作成の場合に備えて作成
BASE_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

import requests


def download_civitai_model(url, output_path, civitai_key):
    headers = {"Authorization": f"Bearer {civitai_key}"}

    with requests.get(url, headers=headers, stream=True, allow_redirects=True, timeout=60) as r:
        print("status:", r.status_code)
        print("final url:", r.url)
        print("content-type:", r.headers.get("content-type"))
        r.raise_for_status()

        content_type = r.headers.get("content-type", "") or ""
        if "text/html" in content_type.lower():
            raise RuntimeError(
                "HTMLが返っています。CIVITAI_KEY、ライセンス同意、短縮URLの解決を確認してください。"
            )

        tmp_path = output_path.with_suffix(output_path.suffix + ".part")
        with open(tmp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        tmp_path.replace(output_path)

    size_mb = output_path.stat().st_size / 1024 / 1024
    if size_mb < 100:
        raise RuntimeError(f"Downloaded file is unexpectedly small: {size_mb:.1f} MB")
    print(f"Downloaded: {output_path}")
    print(f"Size: {size_mb:.1f} MB")

if BASE_MODEL_PATH.exists():
    print(f"Already exists: {BASE_MODEL_PATH}")
    print(f"Size: {BASE_MODEL_PATH.stat().st_size / 1024 / 1024:.1f} MB")
else:
    download_civitai_model(BASE_MODEL_URL, BASE_MODEL_PATH, CIVITAI_KEY)

Already exists: /content/drive/MyDrive/Loras/HibikiMei/models/Sierunami_v1.safetensors
Size: 6776.2 MB


## 4. Install Kohya LTS stack

`kohya-ss/sd-scripts` をタグ固定で取得します。ダウンロードが完了してからインストールを開始します。

開発メモ:ここでコミットバージョンを指定したほうがいいかも


In [6]:
#@title 4.1 Clone sd-scripts
import os
import subprocess
import sys


def run(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=cwd, check=check)

if not KOHYA_DIR.exists():
    run(["git", "clone", KOHYA_REPO, KOHYA_DIR])

run(["git", "fetch", "--tags"], cwd=KOHYA_DIR)
run(["git", "checkout", KOHYA_REF], cwd=KOHYA_DIR)

run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

# PyTorch first, matched to CUDA wheel index.
run([
    sys.executable, "-m", "pip", "install",
    f"torch=={PYTORCH_VERSION}",
    f"torchvision=={TORCHVISION_VERSION}",
    "--index-url", CUDA_INDEX_URL,
])

# Upstream sd-scripts requirements.
run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=KOHYA_DIR)

# xformers is helpful but not mandatory. Do not fail the whole setup if unavailable.
run([sys.executable, "-m", "pip", "install", "xformers", "--index-url", CUDA_INDEX_URL], check=False)

os.chdir(KOHYA_DIR)
print("Kohya directory:", Path.cwd())

$ git clone https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts
$ git fetch --tags
$ git checkout v0.10.5
$ /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
$ /usr/bin/python3 -m pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
$ /usr/bin/python3 -m pip install -r requirements.txt
$ /usr/bin/python3 -m pip install xformers --index-url https://download.pytorch.org/whl/cu124
Kohya directory: /content/sd-scripts


In [7]:
#@title 4.2 Import check
import torch
import accelerate
import transformers
import diffusers
import safetensors

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("accelerate:", accelerate.__version__)
print("transformers:", transformers.__version__)
print("diffusers:", diffusers.__version__)
print("safetensors:", safetensors.__version__)

torch: 2.6.0+cu124
cuda available: True
gpu: Tesla T4
accelerate: 1.6.0
transformers: 4.54.1
diffusers: 0.32.1
safetensors: 0.4.5


## 5. Dataset check

データセットの PNG ファイルを確認し、キャプションがなければ自動生成します。


In [8]:
#@title 5.1 Scan dataset
import re
from PIL import Image

CREATE_MISSING_CAPTIONS = True  #@param {type:"boolean"}
DEFAULT_CAPTION = f"{TRIGGER_WORDS}, illustration"

if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")

png_files = sorted(DATASET_DIR.glob("*.png"), key=lambda p: int(p.stem) if p.stem.isdigit() else 10**12)
if not png_files:
    raise FileNotFoundError(f"No .png files found in: {DATASET_DIR}")

invalid = [p.name for p in png_files if not re.match(r"^[0-9]+\.png$", p.name)]
if invalid:
    raise ValueError("n.png 形式ではないファイルがあります: " + ", ".join(invalid[:20]))

created = 0
for image_path in png_files:
    caption_path = image_path.with_suffix(".txt")
    if CREATE_MISSING_CAPTIONS and not caption_path.exists():
        caption_path.write_text(DEFAULT_CAPTION + "\n", encoding="utf-8")
        created += 1

# Lightweight image validation
bad_images = []
for image_path in png_files:
    try:
        with Image.open(image_path) as im:
            im.verify()
    except Exception as e:
        bad_images.append((image_path.name, str(e)))

if bad_images:
    raise ValueError(f"Broken images detected: {bad_images[:5]}")

steps_per_epoch = len(png_files) * NUM_REPEATS / TRAIN_BATCH_SIZE
estimated_total_steps = steps_per_epoch * MAX_TRAIN_EPOCHS

print(f"Found {len(png_files)} PNG images")
print(f"Created {created} missing caption files")
print(f"Repeats: {NUM_REPEATS}")
print(f"Batch size: {TRAIN_BATCH_SIZE}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Epochs: {MAX_TRAIN_EPOCHS}")
print(f"Estimated total steps: {estimated_total_steps}")


Found 25 PNG images
Created 25 missing caption files
Repeats: 10
Batch size: 2
Steps per epoch: 125.0
Epochs: 10
Estimated total steps: 1250.0


## 6. Generate Kohya config TOML


In [9]:
#@title 6.1 Write TOML configs
for d in [DATASET_DIR, MODEL_DIR, OUTPUT_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "toml"], check=True)
import toml

DATASET_CONFIG_FILE  = CONFIG_DIR / "dataset_config.toml"
TRAINING_CONFIG_FILE = CONFIG_DIR / "training_config.toml"

dataset_config = {
    "general": {
        "shuffle_caption": True,
        "caption_extension": ".txt",
        "keep_tokens": 1,
    },
    "datasets": [{
        "resolution": [1024, 1024],
        "batch_size": TRAIN_BATCH_SIZE,
        "keep_tokens": 1,
        "subsets": [{
            "image_dir": str(DATASET_DIR),
            "num_repeats": NUM_REPEATS,
            "caption_extension": ".txt",
        }],
    }],
}

training_config = {
    "pretrained_model_name_or_path": str(BASE_MODEL_PATH),
    "output_dir": str(OUTPUT_DIR),
    "output_name": PROJECT_NAME,
    "save_model_as": "safetensors",
    "dataset_config": str(DATASET_CONFIG_FILE),
    "network_module": "networks.lora",
    "network_dim": NETWORK_DIM,
    "network_alpha": NETWORK_ALPHA,
    "resolution": RESOLUTION,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "max_train_epochs": MAX_TRAIN_EPOCHS,
    "save_every_n_epochs": 1,
    "learning_rate": LEARNING_RATE,
    "optimizer_type": "AdamW8bit",
    "mixed_precision": "fp16",
    "save_precision": "fp16",
    "cache_latents": True,
    "cache_latents_to_disk": True,
    "gradient_checkpointing": True,
    "xformers": True,
    "lowram": True,
    "max_data_loader_n_workers": 2,
    "persistent_data_loader_workers": True,
    "caption_extension": ".txt",
    "shuffle_caption": True,
    "keep_tokens": 1,
    "logging_dir": str(DRIVE_PROJECT_DIR / "logs"),
}

DATASET_CONFIG_FILE.write_text(toml.dumps(dataset_config), encoding="utf-8")
TRAINING_CONFIG_FILE.write_text(toml.dumps(training_config), encoding="utf-8")

print("Wrote:", DATASET_CONFIG_FILE)
print(DATASET_CONFIG_FILE.read_text(encoding="utf-8"))
print("Wrote:", TRAINING_CONFIG_FILE)


Wrote: /content/drive/MyDrive/Loras/HibikiMei/config/dataset_config.toml
[[datasets]]
resolution = [ 1024, 1024,]
batch_size = 2
keep_tokens = 1
[[datasets.subsets]]
image_dir = "/content/drive/MyDrive/Loras/HibikiMei/dataset"
num_repeats = 10
caption_extension = ".txt"


[general]
shuffle_caption = true
caption_extension = ".txt"
keep_tokens = 1

Wrote: /content/drive/MyDrive/Loras/HibikiMei/config/training_config.toml


## 7. Train


In [10]:
#@title 7.1 Launch training
import subprocess, shlex

train_script = KOHYA_DIR / "sdxl_train_network.py"
if not train_script.exists():
    raise FileNotFoundError(f"Training script not found: {train_script}")

cmd = [
    "accelerate", "launch",
    "--num_cpu_threads_per_process=1",
    str(train_script),
    "--config_file", str(TRAINING_CONFIG_FILE),
]

print("Training command:")
print(" ".join(shlex.quote(str(x)) for x in cmd))
subprocess.run(cmd, check=True, cwd=KOHYA_DIR)


Training command:
accelerate launch --num_cpu_threads_per_process=1 /content/sd-scripts/sdxl_train_network.py --config_file /content/drive/MyDrive/Loras/HibikiMei/config/training_config.toml


CalledProcessError: Command '['accelerate', 'launch', '--num_cpu_threads_per_process=1', '/content/sd-scripts/sdxl_train_network.py', '--config_file', '/content/drive/MyDrive/Loras/HibikiMei/config/training_config.toml']' returned non-zero exit status 1.

## 8. Upload to HuggingFace（省略可）

学習完了後、データセットと LoRA を HuggingFace にアップロードします。
`HF_TOKEN` Colab Secret が必要です。


In [ ]:
#@title 8.1 Upload to HuggingFace
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN が Colab Secrets にありません")

HF_DATASET_REPO = "AICU/HibikiMei"  #@param {type:"string"}
HF_LORA_REPO    = "AICU/SDXL-LoRA"  #@param {type:"string"}

api = HfApi(token=HF_TOKEN)

# --- Upload dataset images ---
UPLOAD_DATASET = True  #@param {type:"boolean"}
if UPLOAD_DATASET:
    png_files = sorted(DATASET_DIR.glob("*.png"))
    print(f"Uploading {len(png_files)} images to datasets/{HF_DATASET_REPO} ...")
    for img in png_files:
        api.upload_file(
            path_or_fileobj=str(img),
            path_in_repo=f"dataset/{img.name}",
            repo_id=HF_DATASET_REPO,
            repo_type="dataset",
            commit_message=f"add {img.name}",
        )
    print("Dataset upload complete:",
          f"https://huggingface.co/datasets/{HF_DATASET_REPO}")

# --- Upload trained LoRA ---
UPLOAD_LORA = True  #@param {type:"boolean"}
if UPLOAD_LORA:
    lora_files = list(OUTPUT_DIR.glob("*.safetensors"))
    if not lora_files:
        print("No .safetensors found in output dir. Run training first.")
    else:
        for lora in lora_files:
            api.upload_file(
                path_or_fileobj=str(lora),
                path_in_repo=f"{PROJECT_NAME}/{lora.name}",
                repo_id=HF_LORA_REPO,
                repo_type="model",
                commit_message=f"{PROJECT_NAME}: add {lora.name}",
            )
        print("LoRA upload complete:",
              f"https://huggingface.co/{HF_LORA_REPO}/tree/main/{PROJECT_NAME}")


## 9. Changelog
